# Notebook-first application walkthrough

**Problem / objective:** Detect data quality, distribution, discrimination and calibration changes after a model is deployed.

**Decision / solution:** Escalate, investigate or retrain only when monitored signals cross documented policy thresholds.

This front section is intentionally analysis-first. It uses direct notebook code for inspection, EDA, visualisation and evidence review. The original notebook work is preserved below, followed by modular production code where that adds engineering evidence.


In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
PROJECT_SLUG = 'model_watch'
ROOT = Path.cwd()
if not (ROOT / 'projects').exists():
    candidate = ROOT.parent.parent if ROOT.name == PROJECT_SLUG else ROOT
    if (candidate / 'projects').exists():
        ROOT = candidate
PROJECT = ROOT / 'projects' / PROJECT_SLUG
if not PROJECT.exists() and Path.cwd().name == PROJECT_SLUG:
    PROJECT = Path.cwd()
    ROOT = PROJECT.parent.parent
assert PROJECT.exists(), f'Project directory not found: {PROJECT}'
print('Repository root:', ROOT.resolve())
print('Project:', PROJECT.resolve())


## 1. Find the real data and retained evidence

Instead of hiding the dataset behind a helper function, start by seeing what the project actually ships: raw/small data, fixtures, outputs, results and verified evidence. External large datasets remain reproducibly downloadable from the documented source.


In [ ]:
candidate_files = []
for pattern in ('*.csv', '*.parquet', '*.json', '*.tsv', '*.txt'):
    candidate_files.extend(PROJECT.rglob(pattern))
verified_dir = ROOT / 'verified' / PROJECT_SLUG
if verified_dir.exists():
    for pattern in ('*.csv', '*.parquet', '*.json', '*.tsv', '*.txt'):
        candidate_files.extend(verified_dir.rglob(pattern))
candidate_files = sorted({p.resolve() for p in candidate_files if p.is_file()})
file_inventory = pd.DataFrame({
    'file': [str(p.relative_to(ROOT)) if ROOT in p.parents else str(p) for p in candidate_files],
    'suffix': [p.suffix.lower() for p in candidate_files],
    'size_kb': [round(p.stat().st_size / 1024, 1) for p in candidate_files],
})
display(file_inventory.head(40))
print(f'Inspectable local data/evidence files: {len(file_inventory):,}')


## 2. Direct tabular data audit

The code below deliberately avoids a project-specific wrapper. It opens the first sensible local tabular asset, shows its schema and quality profile, and makes the data issues visible before modelling. If the full raw dataset is external, run the project's documented download cell/entry point first and rerun this section.


In [ ]:
tabular_candidates = [p for p in candidate_files if p.suffix.lower() in {'.csv', '.tsv', '.parquet'}]
preferred = [p for p in tabular_candidates if not any(token in p.name.lower() for token in ('metric', 'summary', 'verification'))]
tabular_path = (preferred or tabular_candidates or [None])[0]
df = None
if tabular_path is not None:
    if tabular_path.suffix.lower() == '.parquet':
        df = pd.read_parquet(tabular_path)
    else:
        sep = '\t' if tabular_path.suffix.lower() == '.tsv' else ','
        df = pd.read_csv(tabular_path, sep=sep, nrows=200_000)
    print('Loaded:', tabular_path)
    print('Shape:', df.shape)
    display(df.head())
    audit = pd.DataFrame({
        'dtype': df.dtypes.astype(str),
        'missing': df.isna().sum(),
        'missing_pct': (100 * df.isna().mean()).round(2),
        'unique': df.nunique(dropna=False),
    }).sort_values(['missing_pct', 'unique'], ascending=[False, False])
    display(audit.head(30))
    print('Duplicate rows:', int(df.duplicated().sum()))
else:
    print('No local CSV/TSV/Parquet found yet. Use the project README/run path to download or build the documented dataset, then rerun this audit.')


## 3. Exploratory data analysis and visualisation

These plots are intentionally created in the notebook rather than described in prose. They expose distribution, missingness, scale, category balance and numeric relationships before any final model decision.


In [ ]:
if df is not None and len(df):
    missing_pct = (100 * df.isna().mean()).sort_values(ascending=False).head(20)
    missing_pct = missing_pct[missing_pct > 0]
    if len(missing_pct):
        plt.figure(figsize=(10, 4))
        missing_pct.plot(kind='bar')
        plt.title('Missing values by feature (%)')
        plt.ylabel('Missing %')
        plt.xticks(rotation=60, ha='right')
        plt.tight_layout()
        plt.show()

    numeric_cols = df.select_dtypes(include=np.number).columns.tolist()[:8]
    for col in numeric_cols:
        series = pd.to_numeric(df[col], errors='coerce').dropna()
        if len(series):
            plt.figure(figsize=(8, 4))
            plt.hist(series, bins=30, alpha=0.8)
            plt.axvline(series.median(), linestyle='--', label=f'median={series.median():.2f}')
            plt.title(f'Distribution: {col}')
            plt.xlabel(col)
            plt.ylabel('Count')
            plt.legend()
            plt.tight_layout()
            plt.show()

    categorical_cols = [c for c in df.columns if c not in numeric_cols and df[c].nunique(dropna=False) <= 30][:4]
    for col in categorical_cols:
        counts = df[col].fillna('<missing>').astype(str).value_counts().head(15)
        plt.figure(figsize=(9, 4))
        counts.sort_values().plot(kind='barh')
        plt.title(f'Top categories: {col}')
        plt.xlabel('Rows')
        plt.tight_layout()
        plt.show()

    if len(numeric_cols) >= 2:
        corr = df[numeric_cols].corr(numeric_only=True)
        plt.figure(figsize=(8, 6))
        image = plt.imshow(corr, vmin=-1, vmax=1, cmap='coolwarm')
        plt.colorbar(image, label='Correlation')
        plt.xticks(range(len(corr.columns)), corr.columns, rotation=60, ha='right')
        plt.yticks(range(len(corr.index)), corr.index)
        plt.title('Numeric correlation matrix')
        plt.tight_layout()
        plt.show()

    if len(numeric_cols) >= 2:
        x_col, y_col = numeric_cols[0], numeric_cols[-1]
        sample = df[[x_col, y_col]].dropna().sample(min(3000, len(df.dropna(subset=[x_col, y_col]))), random_state=42)
        if len(sample):
            plt.figure(figsize=(7, 5))
            plt.scatter(sample[x_col], sample[y_col], alpha=0.35, s=18)
            plt.xlabel(x_col)
            plt.ylabel(y_col)
            plt.title(f'{y_col} versus {x_col}')
            plt.tight_layout()
            plt.show()
else:
    print('Run the documented data-build/download path, then rerun this section to render raw-data EDA.')


## 4. Inspect the measured results, not just the code

A portfolio project is stronger when it retains evidence. This section reads machine-readable JSON/CSV outputs and turns scalar metrics into a quick visual comparison.


In [ ]:
json_files = [p for p in candidate_files if p.suffix.lower() == '.json']
metric_rows = []
for path in json_files[:30]:
    try:
        payload = json.loads(path.read_text(encoding='utf-8'))
    except Exception:
        continue
    stack = [('', payload)]
    while stack:
        prefix, value = stack.pop()
        if isinstance(value, dict):
            for key, child in value.items():
                stack.append((f'{prefix}.{key}' if prefix else str(key), child))
        elif isinstance(value, (int, float)) and not isinstance(value, bool) and np.isfinite(value):
            metric_rows.append({
                'file': str(path.relative_to(ROOT)) if ROOT in path.parents else str(path),
                'metric': prefix,
                'value': float(value),
            })
metrics_df = pd.DataFrame(metric_rows)
if len(metrics_df):
    display(metrics_df.head(40))
    plot_df = metrics_df[np.isfinite(metrics_df['value'])].copy()
    plot_df = plot_df[plot_df['value'].abs() < 1_000_000].head(20)
    if len(plot_df):
        labels = (plot_df['file'].str.split('/').str[-1] + ' :: ' + plot_df['metric']).tolist()
        plt.figure(figsize=(10, max(4, 0.35 * len(plot_df))))
        plt.barh(range(len(plot_df)), plot_df['value'])
        plt.yticks(range(len(plot_df)), labels)
        plt.title('Retained project metrics / evidence')
        plt.tight_layout()
        plt.show()
else:
    print('No scalar JSON evidence found. Run the project and retain metrics/results before treating it as complete.')


## 5. Reproduce the application

The notebook should be understandable without running anything, but a reviewer can reproduce the canonical application below. The switch is off by default so opening the notebook never triggers a long training job unexpectedly.


In [ ]:
RUN_PROJECT = False
entrypoint = PROJECT / 'run.py'
if RUN_PROJECT and entrypoint.exists():
    subprocess.run([sys.executable, str(entrypoint)], cwd=PROJECT, check=True)
elif entrypoint.exists():
    print(f'Reproduce with: cd {PROJECT} && {sys.executable} run.py')
else:
    print('This project uses a different documented entry point; see README.md in the project folder.')


## 6. Decision / solution

Escalate, investigate or retrain only when monitored signals cross documented policy thresholds.

The final recommendation should be tied to the measured validation evidence and error analysis below. A model is not the solution by itself; the solution is the decision process built around it.


# ModelWatch — Full Python Code

**Hiring purpose:** one project, one notebook, with the actual Python implementation visible. The modular files remain in the repository because that is how production code should be organised; this notebook mirrors those files so a recruiter can inspect the full code without hunting.


## Dataset and reproducibility

Deterministic reference and shifted monitoring batches generated by the project.

The project README/data card documents provenance, constraints and the exact reproduction path. Large third-party raw files are not duplicated in Git when licensing or repository size makes that poor engineering practice.


In [ ]:
from pathlib import Path
import json, os, subprocess, sys

PROJECT_SLUG = 'model_watch'
ROOT = Path.cwd()
if not (ROOT / 'projects').exists():
    target = Path('/content/uni_projects')
    if not target.exists():
        subprocess.run(['git', 'clone', 'https://github.com/Jorgoluka100/uni_projects.git', str(target)], check=True)
    os.chdir(target)
    ROOT = target
PROJECT = ROOT / 'projects' / PROJECT_SLUG
assert PROJECT.exists(), PROJECT
print('Project:', PROJECT.resolve())


## Full Python implementation

Every code cell below is copied directly from the corresponding `.py` file on the same commit. These cells are intentionally tagged `source-mirror` so the notebook acts as a readable code portfolio while the canonical modules remain testable files.


### `run.py`


In [ ]:
from __future__ import annotations

import argparse
import json
import math
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from scipy.stats import ks_2samp
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

FEATURES = ["age", "income", "tenure_months", "usage"]


def generate(n: int, seed: int, shift: float = 0.0, concept: float = 0.0) -> pd.DataFrame:
    if not isinstance(n, int) or n < 100:
        raise ValueError("n must be an integer of at least 100")
    if not np.isfinite([shift, concept]).all() or shift < -9 or shift > 8:
        raise ValueError("shift and concept must be finite; shift must be between -9 and 8")
    rng = np.random.default_rng(seed)
    segment_probability = [0.7 - max(0.0, shift) * 0.08, 0.3 + max(0.0, shift) * 0.08]
    segment = rng.choice(["A", "B"], n, p=segment_probability)
    age = rng.normal(39 + 3 * shift, 11, n).clip(18, 80)
    income = rng.lognormal(np.log(42000 * (1 + 0.10 * shift)), 0.45, n)
    tenure = rng.gamma(2.4 + 0.15 * shift, 12, n).clip(1, 120)
    usage = rng.normal(52 - 5 * shift, 14, n).clip(0, 100)
    logit = (
        -2.2
        + 0.018 * (age - 40)
        + 0.000012 * (income - 42000)
        - 0.018 * (tenure - 24)
        + 0.025 * (usage - 50)
        + 0.55 * (segment == "B")
        + concept * (0.04 * (age - 40) - 0.03 * (usage - 50))
    )
    probability = 1 / (1 + np.exp(-logit))
    target = (rng.random(n) < probability).astype(int)
    if np.unique(target).size != 2:
        raise ValueError("generated sample must contain both target classes; increase n")
    return pd.DataFrame({
        "age": age,
        "income": income,
        "tenure_months": tenure,
        "usage": usage,
        "segment": segment,
        "target": target,
    })


def psi(reference: np.ndarray, current: np.ndarray, bins: int = 10) -> float:
    """Population Stability Index with a defined fallback for constant references."""
    ref = np.asarray(reference, dtype=float)
    cur = np.asarray(current, dtype=float)
    if ref.ndim != 1 or cur.ndim != 1 or ref.size == 0 or cur.size == 0:
        raise ValueError("reference and current must be non-empty 1D arrays")
    if not np.isfinite(ref).all() or not np.isfinite(cur).all():
        raise ValueError("reference and current values must be finite")
    if not isinstance(bins, int) or bins < 2:
        raise ValueError("bins must be an integer of at least two")

    if np.unique(ref).size == 1:
        centre = float(ref[0])
        epsilon = max(abs(centre) * 1e-9, 1e-9)
        edges = np.array([-np.inf, centre - epsilon, centre + epsilon, np.inf])
    else:
        edges = np.unique(np.quantile(ref, np.linspace(0, 1, bins + 1)))
        if edges.size < 3:
            edges = np.linspace(float(ref.min()), float(ref.max()), 3)
        edges[0], edges[-1] = -np.inf, np.inf

    ref_counts = np.histogram(ref, bins=edges)[0].astype(float)
    cur_counts = np.histogram(cur, bins=edges)[0].astype(float)
    ref_share = np.clip(ref_counts / ref_counts.sum(), 1e-6, None)
    cur_share = np.clip(cur_counts / cur_counts.sum(), 1e-6, None)
    return float(np.sum((cur_share - ref_share) * np.log(cur_share / ref_share)))


def ece(y: np.ndarray, p: np.ndarray, bins: int = 10) -> float:
    truth = np.asarray(y)
    probability = np.asarray(p, dtype=float)
    if truth.ndim != 1 or probability.ndim != 1 or truth.size == 0 or truth.shape != probability.shape:
        raise ValueError("target and probability must be non-empty aligned 1D arrays")
    if not np.isin(truth, [0, 1]).all():
        raise ValueError("target must contain binary 0/1 labels")
    if not np.isfinite(probability).all() or np.any((probability < 0) | (probability > 1)):
        raise ValueError("probability must be finite and within [0, 1]")
    if not isinstance(bins, int) or bins < 2:
        raise ValueError("bins must be an integer of at least two")
    truth = truth.astype(int)
    edges = np.linspace(0, 1, bins + 1)
    score = 0.0
    for index, (low, high) in enumerate(zip(edges[:-1], edges[1:])):
        mask = (probability >= low) & (
            probability <= high if index == bins - 1 else probability < high
        )
        if mask.any():
            score += float(mask.mean()) * abs(float(truth[mask].mean() - probability[mask].mean()))
    return float(score)


def subgroup_metrics(df: pd.DataFrame, p: np.ndarray) -> dict:
    required = {"segment", "target"}
    missing = sorted(required - set(df.columns))
    probabilities = np.asarray(p, dtype=float)
    if df.empty or missing or probabilities.ndim != 1 or len(probabilities) != len(df):
        raise ValueError(f"subgroup data must be non-empty, aligned and include {sorted(required)}")
    if not np.isfinite(probabilities).all() or np.any((probabilities < 0) | (probabilities > 1)):
        raise ValueError("subgroup probabilities must be finite and within [0, 1]")
    target = df["target"].to_numpy()
    if not np.isin(target, [0, 1]).all():
        raise ValueError("subgroup target must contain binary 0/1 labels")
    out = {}
    for segment in sorted(df.segment.astype(str).unique()):
        mask = df.segment.astype(str).to_numpy() == segment
        labels = target[mask].astype(int)
        scores = probabilities[mask]
        out[segment] = {
            "rows": int(mask.sum()),
            "prevalence": float(labels.mean()),
            "brier": float(brier_score_loss(labels, scores)),
            "mean_score": float(scores.mean()),
        }
    return out


def evaluate_batch(model, ref: pd.DataFrame, batch: pd.DataFrame, ref_auc: float, name: str) -> dict:
    required = set(FEATURES) | {"target"}
    missing = sorted(required - set(batch.columns))
    missing_reference = sorted(set(FEATURES) - set(ref.columns))
    if batch.empty or ref.empty or missing or missing_reference:
        raise ValueError(f"reference and batch must be non-empty with required columns; batch missing={missing}, reference missing={missing_reference}")
    if not np.isfinite(ref_auc) or not 0 <= ref_auc <= 1:
        raise ValueError("ref_auc must be finite and within [0, 1]")
    y = batch["target"].to_numpy()
    if not np.isin(y, [0, 1]).all() or np.unique(y).size != 2:
        raise ValueError("batch target must contain both binary classes")
    probabilities = np.asarray(model.predict_proba(batch[FEATURES])[:, 1], dtype=float)
    if probabilities.shape != y.shape or not np.isfinite(probabilities).all():
        raise ValueError("model must return one finite probability per batch row")

    drift = {}
    for column in FEATURES:
        reference_values = ref[column].to_numpy(dtype=float)
        current_values = batch[column].to_numpy(dtype=float)
        if not np.isfinite(reference_values).all() or not np.isfinite(current_values).all():
            raise ValueError(f"feature {column} contains non-finite values")
        statistic, p_value = ks_2samp(reference_values, current_values)
        drift[column] = {
            "psi": psi(reference_values, current_values),
            "ks_stat": float(statistic),
            "ks_pvalue": float(p_value),
        }

    auc = float(roc_auc_score(y, probabilities))
    pr_auc = float(average_precision_score(y, probabilities))
    brier = float(brier_score_loss(y, probabilities))
    calibration_error = ece(y, probabilities)
    max_psi = max(value["psi"] for value in drift.values())
    auc_drop = ref_auc - auc
    level = "green"
    reasons = []
    if max_psi >= 0.25 or auc_drop >= 0.08 or calibration_error >= 0.08:
        level = "red"
    elif max_psi >= 0.10 or auc_drop >= 0.04 or calibration_error >= 0.05:
        level = "amber"
    if max_psi >= 0.10:
        reasons.append(f"feature drift PSI={max_psi:.3f}")
    if auc_drop >= 0.04:
        reasons.append(f"ROC-AUC drop={auc_drop:.3f}")
    if calibration_error >= 0.05:
        reasons.append(f"ECE={calibration_error:.3f}")
    return {
        "batch": name,
        "rows": len(batch),
        "drift": drift,
        "max_psi": max_psi,
        "performance": {
            "roc_auc": auc,
            "pr_auc": pr_auc,
            "brier": brier,
            "ece": calibration_error,
            "auc_drop_vs_reference": auc_drop,
        },
        "subgroups": subgroup_metrics(batch, probabilities),
        "alert": level,
        "reasons": reasons,
    }


def run(output_dir: Path, seed: int = 42) -> dict:
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    train = generate(18000, seed)
    reference = generate(7000, seed + 1)
    model = Pipeline([
        ("scale", StandardScaler()),
        ("model", LogisticRegression(max_iter=1000, random_state=seed)),
    ])
    model.fit(train[FEATURES], train.target)
    reference_probability = model.predict_proba(reference[FEATURES])[:, 1]
    reference_auc = float(roc_auc_score(reference.target, reference_probability))

    batches = [
        ("stable", generate(7000, seed + 2, 0.0, 0.0)),
        ("mild_shift", generate(7000, seed + 3, 0.7, 0.0)),
        ("feature_shift", generate(7000, seed + 4, 1.5, 0.2)),
        ("concept_shift", generate(7000, seed + 5, 2.0, 0.8)),
    ]
    reports = [evaluate_batch(model, reference, batch, reference_auc, name) for name, batch in batches]
    model_path = output_dir / "model.joblib"
    joblib.dump(model, model_path)
    reloaded = joblib.load(model_path)
    parity = bool(np.allclose(
        model.predict_proba(reference[FEATURES].head(100)),
        reloaded.predict_proba(reference[FEATURES].head(100)),
    ))
    payload = {
        "project": "ModelWatch",
        "verification_pass": bool(parity and reports[0]["alert"] == "green" and reports[-1]["alert"] == "red"),
        "scope": "deterministic production-monitoring simulation; not a live production system",
        "reference": {
            "rows": len(reference),
            "roc_auc": reference_auc,
            "pr_auc": float(average_precision_score(reference.target, reference_probability)),
            "brier": float(brier_score_loss(reference.target, reference_probability)),
            "ece": ece(reference.target.to_numpy(), reference_probability),
        },
        "batches": reports,
        "model_reload_parity": parity,
        "policy": {
            "amber": "PSI>=0.10 or AUC drop>=0.04 or ECE>=0.05",
            "red": "PSI>=0.25 or AUC drop>=0.08 or ECE>=0.08",
            "action": "red recommends investigation/retraining; this demo never auto-deploys a replacement model",
        },
    }
    (output_dir / "verification.json").write_text(json.dumps(payload, indent=2), encoding="utf-8")
    summary = [
        {"batch": report["batch"], "alert": report["alert"], "max_psi": report["max_psi"], **report["performance"]}
        for report in reports
    ]
    pd.DataFrame(summary).to_csv(output_dir / "batch_summary.csv", index=False)
    print(json.dumps(payload, indent=2))
    return payload


def self_test() -> None:
    result = run(Path("/tmp/modelwatch_selftest"), 123)
    assert result["verification_pass"], result
    assert result["batches"][0]["alert"] == "green"
    assert result["batches"][-1]["alert"] == "red"
    print("ModelWatch self-test passed.")


def main() -> int:
    parser = argparse.ArgumentParser(description="Run the ModelWatch drift and model-risk monitoring simulation.")
    parser.add_argument("--output-dir", type=Path, default=Path("modelwatch_artifacts"))
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--self-test", action="store_true")
    args = parser.parse_args()
    if args.self_test:
        self_test()
        return 0
    result = run(args.output_dir, args.seed)
    return 0 if result["verification_pass"] else 1


if __name__ == "__main__":
    raise SystemExit(main())


## Run the real project

The cell below executes the canonical project entry point rather than a rewritten toy version. Keep `RUN_PIPELINE = False` when you only want to inspect the notebook; change it to `True` to reproduce the project.


In [ ]:
RUN_PIPELINE = False
if RUN_PIPELINE:
    subprocess.run([sys.executable, 'run.py'], cwd=PROJECT, check=True)
else:
    print(f'Reproduce with: cd {PROJECT} && python run.py')


In [ ]:
evidence = []
for folder in (PROJECT / 'results', ROOT / 'verified' / PROJECT_SLUG):
    if folder.exists():
        evidence.extend(sorted(folder.glob('*.json')))
for path in evidence[:5]:
    print('\n---', path.relative_to(ROOT), '---')
    print(path.read_text(encoding='utf-8')[:12000])


## Interview discussion

Be ready to explain the business problem, dataset provenance, cleaning/preprocessing decisions, leakage controls, modelling or analytical choices, evaluation design, limitations, testing strategy and what you would change in production. The key signal is that the notebook, modular source, tests and retained evidence all tell the same story.


# Deeper exploratory analysis and retained evidence

These direct notebook cells extend the initial EDA with data-quality, scale, relationship, output and error diagnostics. They are intentionally visible here rather than hidden behind project helper functions.


In [ ]:
# Extended data-quality scorecard
if df is not None and len(df):
    quality_rows = []
    for col in df.columns:
        series = df[col]
        row = {
            'feature': col,
            'dtype': str(series.dtype),
            'rows': len(series),
            'missing': int(series.isna().sum()),
            'missing_pct': float(100 * series.isna().mean()),
            'unique': int(series.nunique(dropna=False)),
            'unique_pct': float(100 * series.nunique(dropna=False) / max(len(series), 1)),
        }
        if pd.api.types.is_numeric_dtype(series):
            values = pd.to_numeric(series, errors='coerce').dropna()
            if len(values):
                q1, q3 = values.quantile([0.25, 0.75])
                iqr = q3 - q1
                row.update({
                    'mean': float(values.mean()),
                    'median': float(values.median()),
                    'std': float(values.std()),
                    'p05': float(values.quantile(0.05)),
                    'p95': float(values.quantile(0.95)),
                    'skew': float(values.skew()),
                    'iqr_outliers': int(((values < q1 - 1.5*iqr) | (values > q3 + 1.5*iqr)).sum()),
                })
        quality_rows.append(row)
    deep_quality = pd.DataFrame(quality_rows)
    display(deep_quality.sort_values(['missing_pct','unique'], ascending=[False,False]).head(40))
    if 'iqr_outliers' in deep_quality:
        outlier_view = deep_quality.dropna(subset=['iqr_outliers']).sort_values('iqr_outliers', ascending=False).head(15)
        if len(outlier_view):
            plt.figure(figsize=(10,4))
            plt.bar(outlier_view['feature'], outlier_view['iqr_outliers'])
            plt.title('Potential IQR outliers by feature')
            plt.ylabel('Rows')
            plt.xticks(rotation=60, ha='right')
            plt.tight_layout()
            plt.show()
    card = deep_quality.sort_values('unique', ascending=False).head(20)
    plt.figure(figsize=(10,4))
    plt.bar(card['feature'], card['unique'])
    plt.title('Feature cardinality')
    plt.ylabel('Unique values')
    plt.xticks(rotation=60, ha='right')
    plt.tight_layout()
    plt.show()
    print('Constant columns:', deep_quality.loc[deep_quality['unique'] <= 1, 'feature'].tolist())
    print('High-missing columns:', deep_quality.loc[deep_quality['missing_pct'] >= 30, 'feature'].tolist())
    print('Possible identifier columns:', deep_quality.loc[deep_quality['unique_pct'] >= 95, 'feature'].tolist()[:20])
else:
    print('Materialise the documented dataset to run the extended data-quality scorecard.')


In [ ]:
# Numeric distributions, spread and strongest pairwise relationships
if df is not None and len(df):
    numeric_cols = df.select_dtypes(include=np.number).columns.tolist()[:12]
    for col in numeric_cols:
        values = pd.to_numeric(df[col], errors='coerce').dropna()
        if len(values) < 5:
            continue
        clipped = values.clip(values.quantile(0.01), values.quantile(0.99))
        plt.figure(figsize=(8,4))
        plt.hist(clipped, bins=35, alpha=0.82)
        plt.axvline(values.median(), linestyle='--', label=f'median={values.median():.3g}')
        plt.axvline(values.mean(), linestyle=':', label=f'mean={values.mean():.3g}')
        plt.title(f'Distribution: {col} (1st–99th percentile)')
        plt.xlabel(col)
        plt.ylabel('Rows')
        plt.legend()
        plt.tight_layout()
        plt.show()
        plt.figure(figsize=(8,3))
        plt.boxplot(values, vert=False, showfliers=True)
        plt.title(f'Spread / outliers: {col}')
        plt.xlabel(col)
        plt.tight_layout()
        plt.show()
    if len(numeric_cols) >= 2:
        corr = df[numeric_cols].corr(numeric_only=True)
        pairs = []
        for i, left in enumerate(corr.columns):
            for right in corr.columns[i+1:]:
                value = corr.loc[left, right]
                if pd.notna(value):
                    pairs.append({'feature_a': left, 'feature_b': right, 'correlation': float(value), 'abs_correlation': float(abs(value))})
        corr_pairs = pd.DataFrame(pairs).sort_values('abs_correlation', ascending=False) if pairs else pd.DataFrame()
        if len(corr_pairs):
            display(corr_pairs.head(20).round(4))
            for _, pair in corr_pairs.head(4).iterrows():
                sample = df[[pair['feature_a'], pair['feature_b']]].dropna()
                if len(sample) > 3000:
                    sample = sample.sample(3000, random_state=42)
                plt.figure(figsize=(7,5))
                plt.scatter(sample[pair['feature_a']], sample[pair['feature_b']], alpha=0.30, s=16)
                plt.xlabel(pair['feature_a'])
                plt.ylabel(pair['feature_b'])
                plt.title(f"{pair['feature_a']} vs {pair['feature_b']} (r={pair['correlation']:.2f})")
                plt.tight_layout()
                plt.show()
    categorical = [c for c in df.columns if 2 <= df[c].nunique(dropna=False) <= 20][:8]
    for col in categorical:
        counts = df[col].fillna('<missing>').astype(str).value_counts().head(20)
        shares = 100 * counts / counts.sum()
        display(pd.DataFrame({'rows': counts, 'share_pct': shares.round(2)}))
        plt.figure(figsize=(8,4))
        counts.sort_values().plot(kind='barh')
        plt.title(f'Category balance: {col}')
        plt.xlabel('Rows')
        plt.tight_layout()
        plt.show()
else:
    print('Materialise the documented dataset to run distribution diagnostics.')


In [ ]:
# Temporal coverage where date/time fields exist
if df is not None and len(df):
    time_cols = [c for c in df.columns if any(token in str(c).lower() for token in ('date','time','timestamp','datetime'))]
    print('Date/time candidates:', time_cols[:10])
    for col in time_cols[:4]:
        converted = pd.to_datetime(df[col], errors='coerce')
        valid = converted.dropna()
        if len(valid) >= max(10, int(0.25*len(df))):
            print(col, 'range:', valid.min(), '→', valid.max())
            monthly = valid.dt.to_period('M').value_counts().sort_index()
            if len(monthly) > 1:
                plt.figure(figsize=(10,4))
                plt.plot(monthly.index.astype(str), monthly.values, marker='o')
                plt.title(f'Rows over time: {col}')
                plt.ylabel('Rows')
                plt.xticks(rotation=70, ha='right')
                plt.tight_layout()
                plt.show()


## Retained outputs and error analysis

A strong portfolio keeps inspectable evidence. The cells below profile compact result tables and automatically detect prediction-like columns for residual or misclassification analysis.


In [ ]:
# Load compact result/evidence tables
result_tables = []
for base in [PROJECT/'results', PROJECT/'outputs', PROJECT/'artifacts', ROOT/'verified'/PROJECT_SLUG]:
    if not base.exists():
        continue
    for path in sorted(base.rglob('*')):
        if path.is_file() and path.suffix.lower() in {'.csv','.tsv','.parquet'} and path.stat().st_size < 25_000_000:
            try:
                if path.suffix.lower() == '.parquet':
                    table = pd.read_parquet(path)
                else:
                    table = pd.read_csv(path, sep='	' if path.suffix.lower() == '.tsv' else ',')
            except Exception as exc:
                print('Could not read', path.name, '-', exc)
                continue
            result_tables.append((path, table))
            print('
RESULT TABLE:', path.relative_to(ROOT) if ROOT in path.parents else path)
            print('shape=', table.shape)
            display(table.head(15))
            numeric = table.select_dtypes(include=np.number).columns.tolist()[:12]
            if numeric:
                display(table[numeric].describe().T.round(4))
print('Inspectable result tables:', len(result_tables))


In [ ]:
# Automatic regression/classification-style error diagnostics
actual_tokens = ('actual','target','truth','y_true','observed','label')
pred_tokens = ('prediction','predicted','forecast','y_pred')
confidence_tokens = ('confidence','probability','proba','risk','uncertainty')
for path, table in result_tables:
    actual_cols = [c for c in table.columns if any(token in str(c).lower() for token in actual_tokens)]
    pred_cols = [c for c in table.columns if any(token in str(c).lower() for token in pred_tokens)]
    conf_cols = [c for c in table.columns if any(token in str(c).lower() for token in confidence_tokens)]
    if actual_cols and pred_cols and len(table):
        actual_col = actual_cols[0]
        pred_col = next((c for c in pred_cols if c != actual_col), pred_cols[0])
        actual_num = pd.to_numeric(table[actual_col], errors='coerce')
        pred_num = pd.to_numeric(table[pred_col], errors='coerce')
        numeric_mask = actual_num.notna() & pred_num.notna()
        if numeric_mask.sum() >= 10:
            residual = actual_num[numeric_mask] - pred_num[numeric_mask]
            abs_error = residual.abs()
            print('
', path.name, '| MAE=', round(float(abs_error.mean()),5), '| RMSE=', round(float(np.sqrt(np.mean(residual**2))),5), '| bias=', round(float(residual.mean()),5))
            plt.figure(figsize=(7,5))
            plt.scatter(actual_num[numeric_mask], pred_num[numeric_mask], alpha=0.35, s=18)
            lo = min(actual_num[numeric_mask].min(), pred_num[numeric_mask].min())
            hi = max(actual_num[numeric_mask].max(), pred_num[numeric_mask].max())
            plt.plot([lo,hi],[lo,hi], linestyle='--')
            plt.xlabel(str(actual_col))
            plt.ylabel(str(pred_col))
            plt.title(f'Actual vs predicted — {path.name}')
            plt.tight_layout()
            plt.show()
            plt.figure(figsize=(7,4))
            plt.hist(residual, bins=30, alpha=0.82)
            plt.axvline(0, linestyle='--')
            plt.title(f'Residual distribution — {path.name}')
            plt.tight_layout()
            plt.show()
            worst_idx = abs_error.nlargest(min(15,len(abs_error))).index
            cols = list(dict.fromkeys([actual_col,pred_col]+conf_cols[:2]))
            worst = table.loc[worst_idx, cols].copy()
            worst['absolute_error'] = abs_error.loc[worst_idx].values
            display(worst.sort_values('absolute_error', ascending=False))
        else:
            agreement = table[actual_col].astype(str) == table[pred_col].astype(str)
            print('
', path.name, '| classification agreement=', round(float(agreement.mean()),4))
            if (~agreement).any():
                display(table.loc[~agreement, [actual_col,pred_col]+conf_cols[:2]].head(20))
    elif conf_cols:
        for col in conf_cols[:2]:
            values = pd.to_numeric(table[col], errors='coerce').dropna()
            if len(values) >= 10:
                plt.figure(figsize=(7,4))
                plt.hist(values, bins=30, alpha=0.82)
                plt.title(f'{col} distribution — {path.name}')
                plt.tight_layout()
                plt.show()


In [ ]:
# Display retained visual evidence from actual project runs
png_files = []
for base in [PROJECT/'results', PROJECT/'outputs', PROJECT/'artifacts', ROOT/'verified'/PROJECT_SLUG]:
    if base.exists():
        png_files.extend(sorted(base.rglob('*.png')))
print('Retained PNG figures:', len(png_files))
for path in png_files[:12]:
    try:
        image = plt.imread(path)
        plt.figure(figsize=(10,6))
        plt.imshow(image)
        plt.axis('off')
        plt.title(str(path.relative_to(ROOT)) if ROOT in path.parents else path.name)
        plt.tight_layout()
        plt.show()
    except Exception as exc:
        print('Could not display', path.name, '-', exc)


In [ ]:
# Reproducibility and evidence checklist
checks = [
    {'check':'README present', 'status':(PROJECT/'README.md').exists()},
    {'check':'Recruiter notebook present', 'status':(PROJECT/'project_notebook.ipynb').exists()},
    {'check':'Python implementation present', 'status':any(PROJECT.rglob('*.py'))},
    {'check':'Tests present', 'status':(PROJECT/'tests').exists() and any((PROJECT/'tests').rglob('test*.py'))},
    {'check':'Result/evidence files present', 'status':bool(candidate_files)},
    {'check':'Machine-readable JSON evidence', 'status':bool(json_files)},
    {'check':'Retained visual evidence', 'status':bool(png_files)},
]
checklist = pd.DataFrame(checks)
display(checklist)
print('Evidence checklist pass rate:', f"{100*checklist['status'].mean():.1f}%")
print('A failed item is a prompt to strengthen the project, not something to hide.')


# Robustness, slices and decision analysis

A model or pipeline is useful only when we know where it works, where it fails and what action follows. This section adds direct slice analysis, sensitivity checks and a compact decision memo from the evidence already produced by the project.


In [ ]:
# Quantile slices for important numeric variables
if df is not None and len(df):
    numeric_cols = df.select_dtypes(include=np.number).columns.tolist()[:10]
    quantile_rows = []
    for col in numeric_cols:
        values = pd.to_numeric(df[col], errors='coerce')
        valid = values.dropna()
        if len(valid) < 20 or valid.nunique() < 5:
            continue
        quantiles = valid.quantile([0.01,0.05,0.10,0.25,0.50,0.75,0.90,0.95,0.99])
        for q, value in quantiles.items():
            quantile_rows.append({'feature':col, 'quantile':q, 'value':float(value)})
    quantile_table = pd.DataFrame(quantile_rows)
    if len(quantile_table):
        display(quantile_table.pivot(index='feature', columns='quantile', values='value').round(4))
        for col in quantile_table['feature'].unique()[:6]:
            view = quantile_table[quantile_table['feature']==col]
            plt.figure(figsize=(7,4))
            plt.plot(view['quantile'], view['value'], marker='o')
            plt.xlabel('Quantile')
            plt.ylabel(col)
            plt.title(f'Quantile profile: {col}')
            plt.tight_layout()
            plt.show()
else:
    print('Quantile slices become available after the project dataset is materialised.')


In [ ]:
# Missingness and duplication sensitivity
if df is not None and len(df):
    missing_by_row = df.isna().sum(axis=1)
    print('Rows with any missing value:', int((missing_by_row>0).sum()))
    print('Rows with 2+ missing values:', int((missing_by_row>=2).sum()))
    print('Exact duplicate rows:', int(df.duplicated().sum()))
    if missing_by_row.max() > 0:
        plt.figure(figsize=(7,4))
        missing_by_row.value_counts().sort_index().plot(kind='bar')
        plt.title('Missing cells per row')
        plt.xlabel('Missing cells')
        plt.ylabel('Rows')
        plt.tight_layout()
        plt.show()
    duplicated = df.duplicated(keep=False)
    if duplicated.any():
        display(df.loc[duplicated].head(20))
    numeric_cols = df.select_dtypes(include=np.number).columns.tolist()[:10]
    robust_rows = []
    for col in numeric_cols:
        values = pd.to_numeric(df[col], errors='coerce').dropna()
        if len(values) < 20:
            continue
        median = values.median()
        mad = np.median(np.abs(values-median))
        robust_z = 0.6745*(values-median)/(mad if mad else 1.0)
        robust_rows.append({'feature':col, 'median':median, 'mad':mad, 'robust_outliers_abs_z_gt_3_5':int((np.abs(robust_z)>3.5).sum())})
    robust_outliers = pd.DataFrame(robust_rows).sort_values('robust_outliers_abs_z_gt_3_5', ascending=False) if robust_rows else pd.DataFrame()
    if len(robust_outliers):
        display(robust_outliers.round(4))


In [ ]:
# Concentration / imbalance analysis for important categorical dimensions
if df is not None and len(df):
    categorical = [c for c in df.columns if 2 <= df[c].nunique(dropna=False) <= 50][:10]
    concentration_rows = []
    for col in categorical:
        counts = df[col].fillna('<missing>').astype(str).value_counts()
        shares = counts / counts.sum()
        hhi = float((shares**2).sum())
        concentration_rows.append({'feature':col, 'categories':len(counts), 'largest_share':float(shares.iloc[0]), 'top3_share':float(shares.head(3).sum()), 'hhi':hhi})
    concentration = pd.DataFrame(concentration_rows).sort_values('hhi', ascending=False) if concentration_rows else pd.DataFrame()
    if len(concentration):
        display(concentration.round(4))
        plt.figure(figsize=(9,4))
        plt.bar(concentration['feature'], concentration['largest_share'])
        plt.ylabel('Largest category share')
        plt.title('Category concentration / imbalance')
        plt.xticks(rotation=60, ha='right')
        plt.tight_layout()
        plt.show()


In [ ]:
# Rank all retained scalar metrics and highlight likely success/risk signals
metric_records = []
for path in json_files[:60]:
    try:
        payload = json.loads(path.read_text(encoding='utf-8'))
    except Exception:
        continue
    stack = [('', payload)]
    while stack:
        prefix, value = stack.pop()
        if isinstance(value, dict):
            for key, child in value.items():
                stack.append((f'{prefix}.{key}' if prefix else str(key), child))
        elif isinstance(value, (int,float)) and not isinstance(value,bool) and np.isfinite(value):
            metric_records.append({'file':path.name, 'metric':prefix, 'value':float(value)})
all_metrics = pd.DataFrame(metric_records)
if len(all_metrics):
    signal_pattern = 'accuracy|f1|auc|precision|recall|r2|rmse|mae|loss|coverage|review|drift|psi|brier|calibration|revenue|cost|effect|lift|latency|row|reject|duplicate'
    decision_metrics = all_metrics[all_metrics['metric'].str.contains(signal_pattern, case=False, regex=True)].copy()
    if not len(decision_metrics):
        decision_metrics = all_metrics.copy()
    decision_metrics = decision_metrics.drop_duplicates(['file','metric']).reset_index(drop=True)
    display(decision_metrics.head(60).round(6))
    rate_like = decision_metrics[decision_metrics['metric'].str.contains('accuracy|f1|auc|precision|recall|coverage|rate|r2', case=False, regex=True)]
    if len(rate_like):
        bounded = rate_like[(rate_like['value']>=-1)&(rate_like['value']<=1)].head(30)
        if len(bounded):
            plt.figure(figsize=(10,max(5,0.3*len(bounded))))
            plt.barh(range(len(bounded)), bounded['value'])
            plt.yticks(range(len(bounded)), bounded['file']+' :: '+bounded['metric'])
            plt.xlim(min(-0.05,bounded['value'].min()-0.05),1.05)
            plt.title('Retained rate / quality metrics')
            plt.tight_layout()
            plt.show()
    error_like = decision_metrics[decision_metrics['metric'].str.contains('rmse|mae|loss|error|latency|drift|psi|brier', case=False, regex=True)]
    if len(error_like):
        display(error_like.sort_values('value', ascending=False).head(30).round(6))
else:
    print('No retained scalar JSON metrics are available yet.')


In [ ]:
# Inspect artifact sizes — a quick engineering sanity check
artifact_rows = []
for base in [PROJECT/'artifacts', PROJECT/'results', PROJECT/'outputs', ROOT/'verified'/PROJECT_SLUG]:
    if not base.exists():
        continue
    for path in base.rglob('*'):
        if path.is_file():
            artifact_rows.append({'file':str(path.relative_to(ROOT)) if ROOT in path.parents else str(path), 'suffix':path.suffix.lower(), 'size_kb':path.stat().st_size/1024})
artifacts_df = pd.DataFrame(artifact_rows).sort_values('size_kb', ascending=False) if artifact_rows else pd.DataFrame()
if len(artifacts_df):
    display(artifacts_df.head(40).round(2))
    by_type = artifacts_df.groupby('suffix', as_index=False).agg(files=('file','size'), total_kb=('size_kb','sum')).sort_values('total_kb', ascending=False)
    display(by_type.round(2))
    plt.figure(figsize=(8,4))
    plt.bar(by_type['suffix'].replace('', '<none>'), by_type['total_kb'])
    plt.ylabel('Total KB')
    plt.title('Retained evidence by file type')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()
else:
    print('No retained artifacts/results found.')


In [ ]:
# Threshold / coverage trade-off when a result table contains confidence or probability
for path, table in result_tables:
    conf_cols = [c for c in table.columns if any(token in str(c).lower() for token in ('confidence','probability','proba','score','risk'))]
    correct_cols = [c for c in table.columns if 'correct' in str(c).lower()]
    if not conf_cols or not len(table):
        continue
    confidence = pd.to_numeric(table[conf_cols[0]], errors='coerce')
    valid_conf = confidence.notna()
    if valid_conf.sum() < 20:
        continue
    trade_rows = []
    for threshold in np.linspace(float(confidence[valid_conf].quantile(0.10)), float(confidence[valid_conf].quantile(0.90)), 9):
        accepted = valid_conf & (confidence >= threshold)
        row = {'threshold':float(threshold), 'coverage':float(accepted.mean()), 'review_rate':float((valid_conf & ~accepted).sum()/valid_conf.sum()), 'accepted_rows':int(accepted.sum())}
        if correct_cols:
            correctness = table[correct_cols[0]].astype(bool)
            row['accepted_accuracy'] = float(correctness[accepted].mean()) if accepted.any() else np.nan
        trade_rows.append(row)
    trade = pd.DataFrame(trade_rows)
    print('Trade-off table from', path.name, 'using', conf_cols[0])
    display(trade.round(4))
    plt.figure(figsize=(8,4))
    plt.plot(trade['threshold'], trade['coverage'], marker='o', label='coverage')
    if 'accepted_accuracy' in trade:
        plt.plot(trade['threshold'], trade['accepted_accuracy'], marker='o', label='accepted accuracy')
    plt.xlabel('Threshold')
    plt.ylabel('Rate')
    plt.title(f'Threshold trade-off — {path.name}')
    plt.legend()
    plt.tight_layout()
    plt.show()
    break


In [ ]:
# Produce a concise evidence-backed decision memo inside the notebook
project_summary = {
    'project': PROJECT_SLUG,
    'local_data_or_evidence_files': int(len(candidate_files)),
    'result_tables': int(len(result_tables)),
    'json_evidence_files': int(len(json_files)),
    'visual_evidence_files': int(len(png_files)),
    'has_tests': bool((PROJECT/'tests').exists() and any((PROJECT/'tests').rglob('test*.py'))),
    'has_readme': bool((PROJECT/'README.md').exists()),
}
if df is not None:
    project_summary.update({'inspected_rows':int(len(df)), 'inspected_columns':int(df.shape[1]), 'duplicate_rows':int(df.duplicated().sum()), 'missing_cells':int(df.isna().sum().sum())})
summary_table = pd.DataFrame({'item':list(project_summary.keys()), 'value':list(project_summary.values())})
display(summary_table)
print('DECISION PRINCIPLE')
print('1. Use the measured evidence above, not model complexity, to choose the final approach.')
print('2. Inspect the worst slices/failures before making a business or operational recommendation.')
print('3. Keep uncertain, novel or high-impact cases on a review/escalation path where appropriate.')
print('4. Treat the documented limitations as part of the solution, not as boilerplate.')


# Engineering appendix — canonical application source

The analysis and visual evidence come first. The cells below preserve additional canonical Python from this project for reviewers who want to inspect pipelines, APIs, tests, feature code, monitoring and reusable implementation details.


## Canonical source: `src/analysis.py`


In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Iterable

import numpy as np
import pandas as pd


SEVERITY_ORDER = {"green": 0, "amber": 1, "red": 2}


@dataclass(frozen=True)
class MonitoringDecision:
    batch: str
    severity: str
    investigate: bool
    retrain_candidate: bool
    human_approval_required: bool
    reasons: tuple[str, ...]


def calibration_table(
    target: np.ndarray,
    probability: np.ndarray,
    bins: int = 10,
) -> pd.DataFrame:
    target = np.asarray(target, dtype=int)
    probability = np.asarray(probability, dtype=float)
    if target.ndim != 1 or probability.ndim != 1 or target.size == 0 or target.shape != probability.shape:
        raise ValueError("Target and probability must be non-empty aligned 1D arrays")
    if not np.isin(target, [0, 1]).all():
        raise ValueError("Target must contain binary 0/1 labels")
    if not np.isfinite(probability).all() or np.any((probability < 0) | (probability > 1)):
        raise ValueError("Probability values must be finite and within [0, 1]")
    if not isinstance(bins, int) or bins < 2:
        raise ValueError("Use at least two calibration bins")
    frame = pd.DataFrame({"target": target, "probability": probability})
    frame["bin"] = pd.cut(
        frame["probability"],
        bins=np.linspace(0.0, 1.0, bins + 1),
        include_lowest=True,
    )
    grouped = (
        frame.groupby("bin", observed=True)
        .agg(
            rows=("target", "size"),
            observed_rate=("target", "mean"),
            predicted_rate=("probability", "mean"),
        )
        .reset_index()
    )
    grouped["absolute_calibration_gap"] = (
        grouped["observed_rate"] - grouped["predicted_rate"]
    ).abs()
    grouped["bin"] = grouped["bin"].astype(str)
    return grouped


def score_decile_table(
    frame: pd.DataFrame,
    probability: np.ndarray,
    target_col: str = "target",
    deciles: int = 10,
) -> pd.DataFrame:
    if target_col not in frame.columns or frame.empty:
        raise ValueError("score-decile frame must be non-empty and contain the target column")
    probabilities = np.asarray(probability, dtype=float)
    if probabilities.ndim != 1 or probabilities.size != len(frame):
        raise ValueError("probability must be a 1D array aligned with the frame")
    if not np.isfinite(probabilities).all() or np.any((probabilities < 0) | (probabilities > 1)):
        raise ValueError("probability values must be finite and within [0, 1]")
    if not isinstance(deciles, int) or deciles < 2:
        raise ValueError("deciles must be an integer of at least two")
    if not np.isin(frame[target_col].to_numpy(), [0, 1]).all():
        raise ValueError("target column must contain binary 0/1 labels")
    work = frame[[target_col]].copy().reset_index(drop=True)
    work["probability"] = probabilities
    work["score_decile"] = pd.qcut(
        work["probability"],
        q=deciles,
        labels=False,
        duplicates="drop",
    )
    grouped = (
        work.groupby("score_decile", observed=True)
        .agg(
            rows=(target_col, "size"),
            prevalence=(target_col, "mean"),
            mean_score=("probability", "mean"),
            min_score=("probability", "min"),
            max_score=("probability", "max"),
        )
        .reset_index()
        .sort_values("score_decile", ascending=False)
    )
    overall_prevalence = float(work[target_col].mean())
    grouped["lift_vs_population"] = grouped["prevalence"] / max(overall_prevalence, 1e-9)
    return grouped


def feature_drift_table(batch_report: dict) -> pd.DataFrame:
    rows: list[dict[str, float | str]] = []
    for feature, metrics in batch_report.get("drift", {}).items():
        rows.append(
            {
                "feature": str(feature),
                "psi": float(metrics.get("psi", np.nan)),
                "ks_stat": float(metrics.get("ks_stat", np.nan)),
                "ks_pvalue": float(metrics.get("ks_pvalue", np.nan)),
            }
        )
    if not rows:
        return pd.DataFrame(columns=["feature", "psi", "ks_stat", "ks_pvalue"])
    return pd.DataFrame(rows).sort_values(["psi", "ks_stat"], ascending=False)


def batch_trend_table(reports: Iterable[dict]) -> pd.DataFrame:
    rows: list[dict[str, float | str | int]] = []
    for index, report in enumerate(reports):
        performance = report.get("performance", {})
        rows.append(
            {
                "batch_order": int(index),
                "batch": str(report.get("batch", index)),
                "rows": int(report.get("rows", 0)),
                "alert": str(report.get("alert", "unknown")),
                "max_psi": float(report.get("max_psi", np.nan)),
                "roc_auc": float(performance.get("roc_auc", np.nan)),
                "pr_auc": float(performance.get("pr_auc", np.nan)),
                "brier": float(performance.get("brier", np.nan)),
                "ece": float(performance.get("ece", np.nan)),
                "auc_drop_vs_reference": float(
                    performance.get("auc_drop_vs_reference", np.nan)
                ),
            }
        )
    return pd.DataFrame(rows)


def subgroup_gap_table(report: dict) -> pd.DataFrame:
    subgroups = report.get("subgroups", {})
    rows: list[dict[str, float | str | int]] = []
    for subgroup, metrics in subgroups.items():
        rows.append(
            {
                "subgroup": str(subgroup),
                "rows": int(metrics.get("rows", 0)),
                "prevalence": float(metrics.get("prevalence", np.nan)),
                "brier": float(metrics.get("brier", np.nan)),
                "mean_score": float(metrics.get("mean_score", np.nan)),
            }
        )
    table = pd.DataFrame(rows)
    if table.empty:
        return table
    table["prevalence_gap_vs_overall"] = table["prevalence"] - np.average(
        table["prevalence"],
        weights=np.maximum(table["rows"], 1),
    )
    table["score_gap_vs_overall"] = table["mean_score"] - np.average(
        table["mean_score"],
        weights=np.maximum(table["rows"], 1),
    )
    return table.sort_values("rows", ascending=False)


def classify_metric_change(
    value: float,
    amber_threshold: float,
    red_threshold: float,
) -> str:
    if not np.isfinite([value, amber_threshold, red_threshold]).all():
        raise ValueError("metric values and alert thresholds must be finite")
    if amber_threshold < 0 or red_threshold < amber_threshold:
        raise ValueError("thresholds must satisfy 0 <= amber <= red")
    if value >= red_threshold:
        return "red"
    if value >= amber_threshold:
        return "amber"
    return "green"


def worst_severity(levels: Iterable[str]) -> str:
    levels = list(levels)
    unknown = sorted(set(levels) - set(SEVERITY_ORDER))
    if unknown:
        raise ValueError(f"unknown monitoring severities: {unknown}")
    if not levels:
        return "green"
    return max(levels, key=lambda level: SEVERITY_ORDER[level])


def decision_from_report(report: dict) -> MonitoringDecision:
    batch = str(report.get("batch", "unknown"))
    alert = str(report.get("alert", "green"))
    if alert not in SEVERITY_ORDER:
        raise ValueError(f"unknown monitoring severity: {alert}")
    reasons = tuple(str(reason) for reason in report.get("reasons", []))
    investigate = alert in {"amber", "red"}
    retrain_candidate = alert == "red"
    return MonitoringDecision(
        batch=batch,
        severity=alert,
        investigate=investigate,
        retrain_candidate=retrain_candidate,
        human_approval_required=True,
        reasons=reasons,
    )


def retraining_policy_table(reports: Iterable[dict]) -> pd.DataFrame:
    rows = []
    for report in reports:
        decision = decision_from_report(report)
        rows.append(
            {
                "batch": decision.batch,
                "severity": decision.severity,
                "investigate": decision.investigate,
                "retrain_candidate": decision.retrain_candidate,
                "human_approval_required": decision.human_approval_required,
                "reasons": " | ".join(decision.reasons),
            }
        )
    return pd.DataFrame(rows)


def consecutive_alerts(
    reports: Iterable[dict],
    minimum_severity: str = "amber",
) -> int:
    if minimum_severity not in SEVERITY_ORDER:
        raise ValueError(f"unknown minimum severity: {minimum_severity}")
    threshold = SEVERITY_ORDER[minimum_severity]
    count = 0
    for report in reversed(list(reports)):
        level = str(report.get("alert", "green"))
        if level not in SEVERITY_ORDER:
            raise ValueError(f"unknown monitoring severity: {level}")
        if SEVERITY_ORDER[level] >= threshold:
            count += 1
        else:
            break
    return count


def operational_recommendation(reports: list[dict]) -> dict[str, object]:
    if not reports:
        return {
            "status": "no_data",
            "action": "collect monitoring batches",
            "auto_retrain": False,
        }
    latest = reports[-1]
    severity = str(latest.get("alert", "green"))
    amber_streak = consecutive_alerts(reports, "amber")
    red_streak = consecutive_alerts(reports, "red")
    if severity == "red" and red_streak >= 2:
        action = "open incident, investigate data/concept drift, prepare retraining candidate"
    elif severity == "red":
        action = "open incident and investigate before any retraining decision"
    elif severity == "amber" and amber_streak >= 2:
        action = "increase review cadence and investigate persistent degradation"
    elif severity == "amber":
        action = "review the next batch and inspect leading drift features"
    else:
        action = "continue standard monitoring cadence"
    return {
        "status": severity,
        "latest_batch": str(latest.get("batch", "unknown")),
        "consecutive_amber_or_worse": int(amber_streak),
        "consecutive_red": int(red_streak),
        "action": action,
        "auto_retrain": False,
        "human_approval_required": True,
    }


def monitoring_scorecard(reports: list[dict]) -> dict[str, object]:
    trend = batch_trend_table(reports)
    decisions = retraining_policy_table(reports)
    worst = worst_severity(trend["alert"].tolist()) if not trend.empty else "green"
    return {
        "batches_monitored": int(len(reports)),
        "worst_severity": worst,
        "red_batches": int((trend["alert"] == "red").sum()) if not trend.empty else 0,
        "amber_batches": int((trend["alert"] == "amber").sum()) if not trend.empty else 0,
        "max_observed_psi": float(trend["max_psi"].max()) if not trend.empty else 0.0,
        "max_auc_drop": float(trend["auc_drop_vs_reference"].max()) if not trend.empty else 0.0,
        "max_ece": float(trend["ece"].max()) if not trend.empty else 0.0,
        "operational_recommendation": operational_recommendation(reports),
        "decision_table": decisions.to_dict(orient="records"),
    }


# Portfolio depth check

**Meaningful visible code lines after all notebook passes:** 890. The working target for a major application is roughly 1,000 meaningful lines when justified by the problem. This notebook is in/above the working depth range. Line count is never permission to add filler; depth must come from data, analysis, visualisation, modelling/engineering, evaluation, robustness and decision logic.


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import pandas as pd
import pytest

PROJECT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(PROJECT))

from src.analysis import (
    calibration_table,
    classify_metric_change,
    decision_from_report,
    worst_severity,
)


def test_calibration_table_rejects_invalid_probability_inputs():
    with pytest.raises(ValueError, match="within"):
        calibration_table(np.array([0, 1]), np.array([0.2, 1.2]))
    with pytest.raises(ValueError, match="binary"):
        calibration_table(np.array([0, 2]), np.array([0.2, 0.8]))


def test_monitoring_policy_rejects_unknown_severity():
    with pytest.raises(ValueError, match="unknown"):
        worst_severity(["green", "critical"])
    with pytest.raises(ValueError, match="unknown"):
        decision_from_report({"batch": "bad", "alert": "critical"})


def test_alert_thresholds_must_be_ordered_and_finite():
    assert classify_metric_change(0.3, 0.1, 0.25) == "red"
    with pytest.raises(ValueError, match="thresholds"):
        classify_metric_change(0.3, 0.4, 0.2)

from run import ece, generate, psi


def test_constant_reference_distribution_does_not_hide_feature_drift():
    assert psi(np.ones(100), np.full(100, 2.0)) > 0.5


def test_model_watch_generation_and_ece_validate_inputs():
    with pytest.raises(ValueError, match="at least 100"):
        generate(20, 42)
    with pytest.raises(ValueError, match="within"):
        ece(np.array([0, 1]), np.array([-0.1, 1.1]))
